## Acknowledgements

To start, Iâ€™d like to thank **Tong Hui Kang** and **konbu17**.

The CoT data and part of the hyperparameter settings used in this notebook are adapted from Tong Hui Kangâ€™s open-source GitHub repository, while the training code is modified based on konbu17â€™s public notebook.  
- [Tong Hui Kangâ€™s open-source GitHub repository](https://github.com/tonghuikang/nemotron)
- [konbu17â€™s public notebook](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot)

## Overview

In this notebook, I will try to reproduce the method released by Tong Hui Kang and train a model that can reach around **0.85** on the public leaderboard.

However, due to differences in the training platform, I can only make my setup as close as possible to Tong Hui Kangâ€™s original configuration. In addition, I did not use the generated augmentation data in this reproduction. The scripts for generating such augmented data can be found in Tong Hui Kangâ€™s repository.

As a result, the final score of this notebook is expected to be slightly lower than the score of Tong Hui Kangâ€™s currently public model.

## Notes on Training

Also, I am still a beginner in practical LLM training, so there are likely many inefficiencies in my training setup.

Although I tried to speed up the process with **Unsloth**, I only managed to reduce the training time from **10+ hours** to **7+ hours**. By comparison, Tong Hui Kang mentioned when sharing his method publicly that even without using **Tinker**, each of his training runs only took **4+ hours**.

Even so, being able to train a model like this by myself was still a rewarding experience. I hope this notebook can also help others participate more effectively in this competition.

## Mode Selection

In [ ]:
# ============================================================
# MODE SELECTION â€” set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

# ============================================================
# TRAINING STAGE CONTROL
# ============================================================
RUN_SFT = True            # Stage 1: Supervised Fine-Tuning
RUN_GRPO = True           # Stage 2: GRPO Reinforcement Learning (after SFT)

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

SEED = 42

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "RUN_SFT": RUN_SFT,
    "RUN_GRPO": RUN_GRPO,
})

## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton specï¼š", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=True,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth (8-bit).")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

In [ ]:
# ============================================================
# Utility: Brace-balanced \boxed{} extraction
# ============================================================
# The original v6 used re.sub(r'\\boxed\{[^}]*\}', '', cot)
# which BREAKS on nested braces like \boxed{\frac{1}{2}}.
# This parser handles any nesting depth.

def extract_boxed(text):
    """Extract content from \\boxed{...} handling nested braces correctly."""
    idx = text.find("\\boxed{")
    if idx == -1:
        return ""
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        if depth == 0:
            return text[start:i]
    return text[start:]


def remove_boxed(text):
    """Remove all \\boxed{...} occurrences (brace-balanced) from text."""
    result = text
    while "\\boxed{" in result:
        idx = result.find("\\boxed{")
        depth, start = 1, idx + 7
        end = len(result)
        for i in range(start, len(result)):
            if result[i] == '{':
                depth += 1
            elif result[i] == '}':
                depth -= 1
            if depth == 0:
                end = i + 1
                break
        result = result[:idx] + result[end:]
    return result


print("Utility functions defined: extract_boxed(), remove_boxed()")

In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # ============================================================
    # Sophisticated LoRA config for Nemotron-3-Nano-30B-A3B
    # Hybrid architecture: Mamba SSM + Attention + MoE
    # ============================================================
    # Design principles:
    #   1. Stay within competition cap: max_lora_rank = 32 (per-module).
    #   2. Differential rank: allocate capacity where the underlying
    #      matrix is large AND the function is reasoning-critical.
    #   3. Drop lm_head LoRA: massive (vocab x hidden), but the answer
    #      format is already learned via SFT — near-zero ROI for math.
    #   4. Right-size Mamba selective-scan params (dt_proj, x_proj):
    #      their underlying dims are small, r=32 is wasted capacity.
    #   5. RSLoRA scaling (alpha/sqrt(r)) with alpha = 2*r per module.
    #   6. DoRA disabled — magnitude vector + extra fwd pass blew memory
    #      on this 8-bit / 30B-MoE setup. Plain LoRA + RSLoRA only.
    # ============================================================

    LORA_RANK_DEFAULT = 32      # ceiling; rank_pattern overrides per-module
    LORA_ALPHA_DEFAULT = 64     # 2x rank ratio (RSLoRA-friendly)
    LORA_DROPOUT = 0.05         # back to 0.05 — DoRA-driven bump no longer needed

    target_modules = [
        # Attention projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        # Mamba block I/O projections (LARGE — in_proj is the x/z gate)
        "in_proj", "out_proj",
        # Mamba selective-scan params (SMALL underlying dim)
        "x_proj", "dt_proj",
        # MoE expert MLP (gate_proj/up_proj project up; down_proj projects down)
        "gate_proj", "up_proj", "down_proj",
        # NOTE: lm_head intentionally DROPPED — see design principle #3.
    ]

    # Per-module rank: full r=32 on the heavy reasoning paths,
    # smaller ranks on outputs/down-projections and tiny SSM params.
    # Regex matches the suffix on fully-qualified module names like
    # "model.layers.12.self_attn.q_proj" or "...mixer.in_proj".
    rank_pattern = {
        # Attention
        r".*\.q_proj$": 32,   # query — full rank, primary reasoning path
        r".*\.k_proj$": 16,   # key (often GQA-shared, smaller width)
        r".*\.v_proj$": 32,   # value — full rank
        r".*\.o_proj$": 16,   # output projection saturates faster
        # Mamba I/O
        r".*\.in_proj$": 32,  # x/z gate — large, critical
        r".*\.out_proj$": 16, # output mixer — moderate is enough
        # Mamba SSM params (small dims; cap rank by underlying capacity)
        r".*\.x_proj$": 8,    # outputs packed (dt, B, C); small
        r".*\.dt_proj$": 4,   # delta projection; very small dim
        # MoE expert MLP
        r".*\.gate_proj$": 32,
        r".*\.up_proj$": 32,
        r".*\.down_proj$": 16,
    }

    # alpha_pattern keeps alpha = 2 * rank per module so RSLoRA scaling
    # (alpha / sqrt(r)) is consistent across heterogeneous ranks.
    alpha_pattern = {
        r".*\.q_proj$": 64,
        r".*\.k_proj$": 32,
        r".*\.v_proj$": 64,
        r".*\.o_proj$": 32,
        r".*\.in_proj$": 64,
        r".*\.out_proj$": 32,
        r".*\.x_proj$": 16,
        r".*\.dt_proj$": 8,
        r".*\.gate_proj$": 64,
        r".*\.up_proj$": 64,
        r".*\.down_proj$": 32,
    }

    print(f"LoRA Config: default r={LORA_RANK_DEFAULT}, alpha={LORA_ALPHA_DEFAULT}, dropout={LORA_DROPOUT}")
    print(f"Target modules: {target_modules}")
    print(f"Rank pattern (per-module): {rank_pattern}")
    print("Features: RSLoRA + heterogeneous rank allocation (DoRA disabled to avoid OOM)")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK_DEFAULT,
        lora_alpha=LORA_ALPHA_DEFAULT,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        rank_pattern=rank_pattern,
        alpha_pattern=alpha_pattern,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=True,         # alpha / sqrt(r) — needed for heterogeneous r
        use_dora=False,          # disabled: caused OOM on 8-bit / 30B-MoE setup
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")

In [ ]:
# (Old alternative LoRA config removed â€” now using DoRA + RSLoRA in cell above)

In [ ]:
model.print_trainable_parameters()

## Mode A: Train on Kaggle

In [ ]:
if TRAIN_ON_KAGGLE and RUN_SFT:
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

    import pandas as pd
    import random
    import hashlib
    import gc, time
    import re
    import math
    from collections import defaultdict
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig
    from torch.utils.data import DataLoader, Sampler

    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # Quality filtering thresholds
    MIN_COT_LENGTH = 100     # Filter out trivial/empty reasoning
    MAX_COT_LENGTH = 6000    # Filter out degenerate/repetitive CoTs

    df = pd.read_csv(DATASET_PATH)
    print(f"Raw dataset: {len(df)} rows")

    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    # -------------------------------------------------------
    # Build records with FIXED <think> tags + quality filtering + dedup
    # -------------------------------------------------------
    records = []
    record_types = []
    record_cot_lengths = []
    seen_hashes = set()
    skipped = {"no_cot": 0, "too_short": 0, "too_long": 0, "duplicate": 0}

    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])

        # Skip empty/missing CoT
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            skipped["no_cot"] += 1
            continue

        # Quality filter: CoT length
        cot_len = len(cot.strip())
        if cot_len < MIN_COT_LENGTH:
            skipped["too_short"] += 1
            continue
        if cot_len > MAX_COT_LENGTH:
            skipped["too_long"] += 1
            continue

        # Deduplication by prompt hash
        prompt_hash = hashlib.md5(prompt.strip().lower().encode()).hexdigest()
        if prompt_hash in seen_hashes:
            skipped["duplicate"] += 1
            continue
        seen_hashes.add(prompt_hash)

        # FIX: Use brace-balanced removal instead of broken regex
        cot_cleaned = remove_boxed(cot).rstrip()

        user_content = prompt + PROMPT_SUFFIX

        # FIX: Include opening <think> tag (was missing in v6!)
        assistant_content = f"<think>\n{cot_cleaned}\n</think>\n\\boxed{{{answer}}}"

        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
        record_types.append(str(row.get("type", "unknown")))
        record_cot_lengths.append(cot_len)

    print(f"\nSFT records after filtering: {len(records)}")
    print(f"Skipped: {skipped}")
    print(f"CoT length stats: min={min(record_cot_lengths)}, max={max(record_cot_lengths)}, "
          f"mean={sum(record_cot_lengths)/len(record_cot_lengths):.0f}")

    # -------------------------------------------------------
    # Curriculum learning: sort by CoT length (easy -> hard)
    # -------------------------------------------------------
    sorted_indices = sorted(range(len(records)), key=lambda i: record_cot_lengths[i])
    records = [records[i] for i in sorted_indices]
    record_types = [record_types[i] for i in sorted_indices]
    record_cot_lengths = [record_cot_lengths[i] for i in sorted_indices]

    print(f"\nCurriculum ordering: shortest CoT first -> longest last")
    print(f"First 5 CoT lengths: {record_cot_lengths[:5]}")
    print(f"Last 5 CoT lengths: {record_cot_lengths[-5:]}")

    # -------------------------------------------------------
    # Train/validation split (5% holdout)
    # -------------------------------------------------------
    VAL_FRAC = 0.05
    n_val = max(1, int(len(records) * VAL_FRAC))
    rng = random.Random(SEED)
    val_indices = set(rng.sample(range(len(records)), n_val))
    train_indices = [i for i in range(len(records)) if i not in val_indices]

    train_records = [records[i] for i in train_indices]
    val_records = [records[i] for i in sorted(val_indices)]
    train_types = [record_types[i] for i in train_indices]

    train_dataset = HFDataset.from_list(train_records)
    val_dataset = HFDataset.from_list(val_records)

    print(f"\nTrain: {len(train_records)}, Validation: {len(val_records)}")
    print("Type distribution:", dict(sorted(pd.Series(train_types).value_counts().to_dict().items())))

    # -------------------------------------------------------
    # Chat template formatting
    # -------------------------------------------------------
    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages
        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=True,
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            texts.append(text)
        return texts

    # -------------------------------------------------------
    # IMPROVED Training config
    # -------------------------------------------------------
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",

        # Epochs & batching
        num_train_epochs=1,                          # Reduced to 1 to avoid OOM
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,               # effective batch size = 8

        # Learning rate
        learning_rate=5e-5,
        lr_scheduler_type="cosine_with_restarts",
        warmup_ratio=0.10,
        lr_scheduler_kwargs={"num_cycles": 2},

        # Sequence length
        max_length=8192,

        # Optimizer
        optim="paged_adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.01,
        max_grad_norm=1.0,

        # NEFTune
        neftune_noise_alpha=5.0,

        # Evaluation
        eval_strategy="steps",
        eval_steps=100,

        # Logging & saving
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,

        # Precision & memory
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
        dataset_num_proc=4,

        # Load best model at end based on eval loss
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    # -------------------------------------------------------
    # Stratified batching (preserved from v6)
    # -------------------------------------------------------
    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)
        rng_local = random.Random(seed)
        for idx_list in by_label.values():
            rng_local.shuffle(idx_list)
        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng_local.shuffle(batch_order)
        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1
        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)
        def __iter__(self):
            return iter(self.order)
        def __len__(self):
            return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order
        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            if len(self.stratified_order) != len(self.train_dataset):
                raise ValueError("Stratified order length does not match train dataset")
            dataloader_kwargs = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
            return DataLoader(self.train_dataset, **dataloader_kwargs)

    effective_batch_size = max(
        1,
        training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    )
    stratified_order = build_stratified_index_order(train_types, effective_batch_size, SEED)
    print(f"\nEffective batch size: {effective_batch_size}")
    print("Stratified batching by type:", dict(sorted(pd.Series(train_types).value_counts().to_dict().items())))

    trainer = StratifiedSFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
    )

    torch.cuda.empty_cache()
    gc.collect()

    print("=" * 60)
    print("Starting Stage 1: SFT Training")
    print(f"  DoRA: OFF (OOM) | NEFTune alpha: 5.0 | LR: 5e-5")
    print(f"  Default rank: {LORA_RANK_DEFAULT} | Default alpha: {LORA_ALPHA_DEFAULT} | RSLoRA: ON")
    print(f"  Heterogeneous rank_pattern active (q/v/in/gate/up = 32; k/o/out/down = 16; x = 8; dt = 4)")
    print(f"  Epochs: 1 | Scheduler: cosine_with_restarts (2 cycles)")
    print(f"  Model: 8-bit quantized")
    print(f"  Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
    print("=" * 60)

    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"\nSFT training done in {elapsed/60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"SFT adapter saved to {ADAPTER_DIR}")

## Stage 2: GRPO (Group Relative Policy Optimization)
**Why GRPO after SFT?**
- SFT teaches the model the *format* (think + boxed answer) and general reasoning
- GRPO teaches the model to *get the right answer* by rewarding correct `\boxed{}` matches
- Generates N completions per prompt, ranks by reward, updates policy
- Additional format reward for proper `<think>...</think>` structure

In [ ]:
# ============================================================
# Stage 2: GRPO Reinforcement Learning
# ============================================================
if TRAIN_ON_KAGGLE and RUN_GRPO:
    import gc
    import time
    import torch
    import pandas as pd
    from datasets import Dataset as HFDataset

    grpo_df = pd.read_csv(DATASET_PATH)
    print(f"GRPO dataset: {len(grpo_df)} problems")

    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    grpo_records = []
    for _, row in grpo_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        if not answer or answer == "nan":
            continue
        grpo_records.append({
            "prompt": prompt + PROMPT_SUFFIX,
            "ground_truth": answer.strip(),
        })

    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f"GRPO records: {len(grpo_records)}")

    # -------------------------------------------------------
    # Reward functions
    # -------------------------------------------------------
    def accuracy_reward(completions, ground_truth, **kwargs):
        """Primary reward: correct \\boxed{} answer = 1.0, wrong = 0.0"""
        rewards = []
        for completion, gt in zip(completions, ground_truth):
            predicted = extract_boxed(completion).strip()
            expected = gt.strip()
            if predicted == expected:
                rewards.append(1.0)
                continue
            try:
                pred_num = float(predicted)
                exp_num = float(expected)
                if abs(pred_num - exp_num) < 1e-2:
                    rewards.append(1.0)
                    continue
            except (ValueError, TypeError):
                pass
            rewards.append(0.0)
        return rewards

    def format_reward(completions, **kwargs):
        """Secondary reward: +0.2 for <think>...</think>, +0.1 for \\boxed{}"""
        rewards = []
        for completion in completions:
            score = 0.0
            if "<think>" in completion and "</think>" in completion:
                think_start = completion.find("<think>")
                think_end = completion.find("</think>")
                if think_start < think_end:
                    score += 0.2
            if "\\boxed{" in completion:
                score += 0.1
            rewards.append(score)
        return rewards

    print("Reward functions defined: accuracy_reward (0/1), format_reward (0-0.3)")

    # -------------------------------------------------------
    # GRPO Training
    # -------------------------------------------------------
    from trl import GRPOTrainer, GRPOConfig

    FastLanguageModel.for_training(model)

    grpo_config = GRPOConfig(
        output_dir="/kaggle/working/grpo_output",

        # GRPO-specific
        num_generations=4,
        max_completion_length=4096,
        max_prompt_length=2048,
        beta=0.04,                    # KL penalty

        # Training
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,

        # Optimizer
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.5,            # Tighter clipping for RL stability

        # Precision & memory
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},

        # Logging
        logging_steps=5,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        train_dataset=grpo_dataset,
        processing_class=tokenizer,
        reward_funcs=[accuracy_reward, format_reward],
    )

    torch.cuda.empty_cache()
    gc.collect()

    print("=" * 60)
    print("Starting Stage 2: GRPO Training")
    print(f"  Generations per prompt: 4 | Beta (KL): 0.04")
    print(f"  LR: 1e-5 | Max completion: 4096 tokens")
    print(f"  Rewards: accuracy (0/1) + format (0-0.3)")
    print(f"  Problems: {len(grpo_dataset)}")
    print("=" * 60)

    t0 = time.time()
    grpo_trainer.train()
    elapsed = time.time() - t0
    print(f"\nGRPO training completed in {elapsed/60:.1f} min")

    GRPO_ADAPTER_DIR = "/kaggle/working/grpo_adapter"
    model.save_pretrained(GRPO_ADAPTER_DIR)
    tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
    print(f"GRPO adapter saved to {GRPO_ADAPTER_DIR}")

## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

# Determine source adapter directory
if TRAIN_ON_KAGGLE:
    if RUN_GRPO:
        src_adapter_dir = "/kaggle/working/grpo_adapter"
        print("Packaging GRPO-trained adapter (SFT + RL)")
    else:
        src_adapter_dir = "/kaggle/working/sft_adapter"
        print("Packaging SFT-trained adapter")
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0  # Disable dropout at inference

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")